# 02 — Player performance: skill, economics, hero pool

Per-player profile over the tier-1 season: win rate, KDA, GPM/XPM (economy),
lane efficiency, teamfight participation, vision, and hero-pool breadth
(`hero_pool`, plus HHI concentration — low HHI = wide comfort pool).
Roles are inferred from within-team economy rank (1=carry … 5=hard support).

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
pd.set_option('display.max_columns', 60); pd.set_option('display.width', 160)
DATA = ROOT / 'data'


In [ ]:
from src.features import player_features
matches = pd.read_parquet(DATA / 'matches.parquet')
mp = pd.read_parquet(DATA / 'match_players.parquet')
pf = player_features(mp, matches, min_games=8)
print(f'{len(pf)} players with >=8 tier-1 games')
pf.head(10)

In [ ]:
# Top players by role: economy vs impact
for role in ['carry','mid','offlane','soft_support','hard_support']:
    top = pf[pf.primary_role==role].nlargest(8,'winrate')
    print(f'\n== {role} (top-8 by winrate) ==')
    print(top[['name','games','winrate','gpm_mean','xpm_mean','kda_mean','hero_pool','hero_hhi']].to_string(index=False))

In [ ]:
# Hero-pool breadth vs winrate — is versatility rewarded this season?
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12,4))
core = pf[pf.primary_role.isin(['carry','mid','offlane'])]
ax[0].scatter(core.hero_pool_per_game, core.winrate, alpha=.5)
ax[0].set(xlabel='unique heroes per game', ylabel='winrate', title='Cores: hero-pool breadth vs winrate')
ax[1].scatter(pf.gpm_mean, pf.kda_mean, c=(pf.winrate>0.55), alpha=.5)
ax[1].set(xlabel='GPM', ylabel='KDA', title='Economy vs impact (dark = winrate>55%)')
plt.tight_layout()

In [ ]:
# Per-team player table (join rosters as-of the season)
team_of = mp.dropna(subset=['team_id']).groupby('account_id').team_id.agg(lambda s: s.mode().iat[0])
pf2 = pf.merge(team_of.rename('team_id'), on='account_id', how='left')
teams = pd.read_parquet(DATA / 'teams.parquet')[['team_id','name']].rename(columns={'name':'team'})
pf2.merge(teams, on='team_id').sort_values(['team','gpm_mean'], ascending=[True,False]).head(30)